# BreakupModel — Computational Time Benchmark

This notebook loads CSV files produced by `benchmark.py` and plots:

1. **Single-branch sweep** — time vs. `minimalCharacteristicLength`  
2. **Cross-branch comparison** — overlay two branches on the same axes  
3. **Fragment count** — how many fragments are generated per run  

---
### How to generate the CSVs

```bash
# Branch A (current build, no git switch needed)
python benchmark.py \
    --repo   /home/andrea/LSMS_project/NASA-breakup-model-cpp \
    --data   /home/andrea/LSMS_project/NASA-breakup-model-cpp/example_config/CTwrtMinCT/data.yaml \
    --output results_branch_A.csv \
    --label  "branch_A"

# Branch B (different git branch — will checkout + rebuild automatically)
python benchmark.py \
    --repo    /home/andrea/LSMS_project/NASA-breakup-model-cpp \
    --data    /home/andrea/LSMS_project/NASA-breakup-model-cpp/example_config/CTwrtMinCT/data.yaml \
    --output  results_branch_B.csv \
    --label   "branch_B" \
    --branch  my-other-branch \
    --rebuild
```


In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────
# pip install pandas matplotlib numpy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

plt.rcParams.update({
    "figure.dpi": 140,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "font.size": 11,
})

## 1 — Configuration

Set the paths to the CSV files generated by `benchmark.py`.

In [ ]:
# ── Point these to your CSV files ────────────────────────────────────────
CSV_A = Path("results_branch_A.csv")   # first branch
CSV_B = Path("results_branch_B.csv")   # second branch (set to None to skip comparison)

# Directory where plots will be saved (set to None to skip saving)
PLOT_DIR = Path("plots")

# ── Colours ───────────────────────────────────────────────────────────────
COLOR_A = "#1f77b4"   # blue
COLOR_B = "#d62728"   # red

## 2 — Load data

In [ ]:
def load_csv(path: Path) -> pd.DataFrame | None:
    if path is None or not path.exists():
        print(f"  [skip] {path} not found.")
        return None
    df = pd.read_csv(path)
    df = df.sort_values("min_cl").reset_index(drop=True)
    print(f"  Loaded {len(df)} rows from {path}  (label: {df['label'].iloc[0]})")
    return df

df_a = load_csv(CSV_A)
df_b = load_csv(CSV_B) if CSV_B is not None else None

if df_a is not None:
    display(df_a)

In [ ]:
# Helper: save figure if PLOT_DIR is set
if PLOT_DIR is not None:
    PLOT_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name: str):
    if PLOT_DIR is not None:
        p = PLOT_DIR / name
        fig.savefig(p, bbox_inches="tight")
        print(f"  Saved → {p}")

## 3 — Single-branch: time vs. minCL

In [ ]:
def plot_single(df: pd.DataFrame, color: str = COLOR_A) -> plt.Figure:
    label = df["label"].iloc[0]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    for ax, xscale, title in zip(
        axes,
        ["linear", "log"],
        ["Linear x-axis", "Log x-axis"],
    ):
        ax.errorbar(
            df["min_cl"], df["mean_ms"],
            yerr=df["std_ms"],
            fmt="o-", color=color, capsize=4,
            linewidth=1.8, markersize=6, label=label,
        )
        ax.fill_between(
            df["min_cl"], df["min_ms"], df["max_ms"],
            alpha=0.15, color=color,
        )
        ax.set_xscale(xscale)
        ax.set_xlabel("minimalCharacteristicLength  [m]")
        ax.set_ylabel("Simulation time  [ms]")
        ax.set_title(title)
        ax.legend()

    fig.suptitle(f"Computational time vs. minCL — {label}", fontsize=13, y=1.01)
    plt.tight_layout()
    return fig


if df_a is not None:
    fig = plot_single(df_a)
    save_fig(fig, "single_branch_A.png")
    plt.show()

## 4 — Single-branch: fragments vs. minCL

In [ ]:
def plot_fragments(df: pd.DataFrame, color: str = COLOR_A) -> plt.Figure:
    label = df["label"].iloc[0]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    for ax, xscale in zip(axes, ["linear", "log"]):
        ax.plot(
            df["min_cl"], df["fragments"],
            "s-", color=color, linewidth=1.8, markersize=6, label=label,
        )
        ax.set_xscale(xscale)
        ax.set_yscale("log")
        ax.set_xlabel("minimalCharacteristicLength  [m]")
        ax.set_ylabel("Fragment count")
        ax.set_title(f"{'Linear' if xscale == 'linear' else 'Log'} x-axis, log y-axis")
        ax.legend()

    fig.suptitle(f"Fragment count vs. minCL — {label}", fontsize=13, y=1.01)
    plt.tight_layout()
    return fig


if df_a is not None:
    fig = plot_fragments(df_a)
    save_fig(fig, "fragments_branch_A.png")
    plt.show()

## 5 — Time vs. fragment count (efficiency curve)

In [ ]:
def plot_time_vs_frags(df: pd.DataFrame, color: str = COLOR_A) -> plt.Figure:
    label = df["label"].iloc[0]
    fig, ax = plt.subplots(figsize=(7, 4.5))

    sc = ax.scatter(
        df["fragments"], df["mean_ms"],
        c=np.log10(df["min_cl"]),
        cmap="plasma", s=70, zorder=3,
    )
    ax.plot(df["fragments"], df["mean_ms"],
            "-", color=color, alpha=0.4, linewidth=1.2)
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("log₁₀(minCL)")

    ax.set_xlabel("Fragment count")
    ax.set_ylabel("Simulation time  [ms]")
    ax.set_title(f"Efficiency: time vs. fragments — {label}")
    plt.tight_layout()
    return fig


if df_a is not None and df_a["fragments"].notna().any():
    fig = plot_time_vs_frags(df_a)
    save_fig(fig, "efficiency_branch_A.png")
    plt.show()

---
## 6 — Cross-branch comparison

Requires both `CSV_A` and `CSV_B` to be available.

In [ ]:
def plot_comparison_time(df_a: pd.DataFrame, df_b: pd.DataFrame) -> plt.Figure:
    label_a = df_a["label"].iloc[0]
    label_b = df_b["label"].iloc[0]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, xscale in zip(axes, ["linear", "log"]):
        for df, color, label in [
            (df_a, COLOR_A, label_a),
            (df_b, COLOR_B, label_b),
        ]:
            ax.errorbar(
                df["min_cl"], df["mean_ms"],
                yerr=df["std_ms"],
                fmt="o-", color=color, capsize=4,
                linewidth=1.8, markersize=6, label=label,
            )
            ax.fill_between(
                df["min_cl"], df["min_ms"], df["max_ms"],
                alpha=0.12, color=color,
            )
        ax.set_xscale(xscale)
        ax.set_xlabel("minimalCharacteristicLength  [m]")
        ax.set_ylabel("Simulation time  [ms]")
        ax.set_title(f"{'Linear' if xscale == 'linear' else 'Log'} x-axis")
        ax.legend()

    fig.suptitle(f"Computational time comparison: {label_a} vs. {label_b}",
                 fontsize=13, y=1.01)
    plt.tight_layout()
    return fig


if df_a is not None and df_b is not None:
    fig = plot_comparison_time(df_a, df_b)
    save_fig(fig, "comparison_time.png")
    plt.show()
else:
    print("Skipping comparison plot (need both CSV_A and CSV_B).")

In [ ]:
def plot_speedup(df_a: pd.DataFrame, df_b: pd.DataFrame) -> plt.Figure:
    """Speedup of branch_B relative to branch_A (>1 means B is faster)."""
    label_a = df_a["label"].iloc[0]
    label_b = df_b["label"].iloc[0]

    # Align on shared minCL values
    merged = pd.merge(
        df_a[["min_cl", "mean_ms"]].rename(columns={"mean_ms": "ms_a"}),
        df_b[["min_cl", "mean_ms"]].rename(columns={"mean_ms": "ms_b"}),
        on="min_cl",
    )
    merged["speedup"] = merged["ms_a"] / merged["ms_b"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    for ax, xscale in zip(axes, ["linear", "log"]):
        ax.axhline(1.0, color="grey", linestyle="--", linewidth=1, label="no change")
        ax.plot(
            merged["min_cl"], merged["speedup"],
            "D-", color="#2ca02c", linewidth=1.8, markersize=6,
            label=f"{label_a} / {label_b}",
        )
        ax.set_xscale(xscale)
        ax.set_xlabel("minimalCharacteristicLength  [m]")
        ax.set_ylabel(f"Speedup  ({label_b} relative to {label_a})")
        ax.set_title(f"{'Linear' if xscale == 'linear' else 'Log'} x-axis")
        ax.legend()

    fig.suptitle(f"Relative speedup: {label_b} vs. {label_a}  (>1 = B faster)",
                 fontsize=13, y=1.01)
    plt.tight_layout()
    return fig


if df_a is not None and df_b is not None:
    fig = plot_speedup(df_a, df_b)
    save_fig(fig, "comparison_speedup.png")
    plt.show()
else:
    print("Skipping speedup plot (need both CSV_A and CSV_B).")

In [ ]:
def plot_comparison_fragments(df_a: pd.DataFrame, df_b: pd.DataFrame) -> plt.Figure:
    label_a = df_a["label"].iloc[0]
    label_b = df_b["label"].iloc[0]

    fig, ax = plt.subplots(figsize=(8, 4.5))

    for df, color, label in [
        (df_a, COLOR_A, label_a),
        (df_b, COLOR_B, label_b),
    ]:
        ax.plot(
            df["min_cl"], df["fragments"],
            "s-", color=color, linewidth=1.8,
            markersize=6, label=label,
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("minimalCharacteristicLength  [m]")
    ax.set_ylabel("Fragment count")
    ax.set_title(f"Fragment count comparison: {label_a} vs. {label_b}")
    ax.legend()
    plt.tight_layout()
    return fig


if df_a is not None and df_b is not None:
    fig = plot_comparison_fragments(df_a, df_b)
    save_fig(fig, "comparison_fragments.png")
    plt.show()
else:
    print("Skipping fragment comparison plot (need both CSV_A and CSV_B).")

---
## 7 — Summary table

In [ ]:
frames = [df for df in [df_a, df_b] if df is not None]
if frames:
    combined = pd.concat(frames, ignore_index=True)
    pivot = combined.pivot_table(
        index="min_cl",
        columns="label",
        values=["mean_ms", "std_ms", "fragments"],
    ).round(1)
    display(pivot)
else:
    print("No data loaded.")